# BÀI 4. XÂY DỰNG MÔ HÌNH MLP CHO DỮ LIỆU ẢNH
## Phần Bài Tập

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import confusion_matrix

# Fallback tự động tải dữ liệu nếu chạy riêng phần bài tập mà chưa chạy các cell phía trên
if "X_train" not in globals():
    (X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
    X_train = X_train_full[:-10_000].astype("float32") / 255.0
    y_train = y_train_full[:-10_000]
    X_val = X_train_full[-10_000:].astype("float32") / 255.0
    y_val = y_train_full[-10_000:]
    X_test = X_test.astype("float32") / 255.0
    class_names = [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
    ]

def run_experiment(hidden_units=[256, 128], lr=1e-3, batch_size=64,
                   dropout_rate=0.0, l2_reg=0.0, epochs=15, verbose=0):
    tf.keras.utils.set_random_seed(42)
    reg = tf.keras.regularizers.l2(l2_reg) if l2_reg > 0 else None

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28)),
        tf.keras.layers.Flatten()
    ])
    for units in hidden_units:
        model.add(tf.keras.layers.Dense(units, activation="relu", kernel_regularizer=reg))
        if dropout_rate > 0:
            model.add(tf.keras.layers.Dropout(dropout_rate))
    model.add(tf.keras.layers.Dense(10, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose
    )
    train_time = time.time() - start_time
    _, test_acc = model.evaluate(X_test, y_test, verbose=0)

    return {
        "model": model,
        "history": history.history,
        "params": model.count_params(),
        "time": train_time,
        "val_acc": max(history.history["val_accuracy"]),
        "test_acc": test_acc
    }

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import confusion_matrix

# Fallback tự động tải dữ liệu nếu chạy riêng phần bài tập mà chưa chạy các cell phía trên
if "X_train" not in globals():
    (X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
    X_train = X_train_full[:-10_000].astype("float32") / 255.0
    y_train = y_train_full[:-10_000]
    X_val = X_train_full[-10_000:].astype("float32") / 255.0
    y_val = y_train_full[-10_000:]
    X_test = X_test.astype("float32") / 255.0
    class_names = [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
    ]

def run_experiment(hidden_units=[256, 128], lr=1e-3, batch_size=64,
                   dropout_rate=0.0, l2_reg=0.0, epochs=15, verbose=0):
    tf.keras.utils.set_random_seed(42)
    reg = tf.keras.regularizers.l2(l2_reg) if l2_reg > 0 else None

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28)),
        tf.keras.layers.Flatten()
    ])
    for units in hidden_units:
        model.add(tf.keras.layers.Dense(units, activation="relu", kernel_regularizer=reg))
        if dropout_rate > 0:
            model.add(tf.keras.layers.Dropout(dropout_rate))
    model.add(tf.keras.layers.Dense(10, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose
    )
    train_time = time.time() - start_time
    _, test_acc = model.evaluate(X_test, y_test, verbose=0)

    return {
        "model": model,
        "history": history.history,
        "params": model.count_params(),
        "time": train_time,
        "val_acc": max(history.history["val_accuracy"]),
        "test_acc": test_acc
    }

---
### Bài 1. Thay đổi kiến trúc (1, 2, 3 hidden layers)

In [ ]:
architectures = {
    "1 hidden layer [128]": [128],
    "2 hidden layers [256, 128]": [256, 128],
    "3 hidden layers [256, 128, 64]": [256, 128, 64],
}

results_b1 = {}
print(f"{'Kiến trúc':<32} | {'Params':>10} | {'Time (s)':>8} | {'Best Val Acc':>12} | {'Test Acc':>10}")
print("-" * 82)

for name, units in architectures.items():
    res = run_experiment(hidden_units=units, epochs=15)
    results_b1[name] = res
    print(f"{name:<32} | {res['params']:>10,} | {res['time']:>8.2f} | {res['val_acc']:>12.4f} | {res['test_acc']:>10.4f}")

**Nhận xét Bài 1:**
* **Số parameter:** Tăng mạnh theo số tầng ẩn và số neuron của tầng đầu tiên ngay sau `Flatten()` (`101,770` tham số với `[128]` $\rightarrow$ `235,146` tham số với `[256, 128]` $\rightarrow$ `242,762` tham số với `[256, 128, 64]`).
* **Thời gian train:** Tăng tỷ lệ thuận với số lượng tham số do khối lượng phép nhân ma trận lớn hơn.
* **Validation & Test accuracy:** Mô hình 2 hidden layers `[256, 128]` cải thiện rõ rệt so với 1 hidden layer vì học được các đặc trưng phi tuyến phức tạp hơn. Khi tăng lên 3 hidden layers mà không có regularization, độ chính xác cải thiện không đáng kể (thậm chí dễ giảm nhẹ trên tập test) do dư thừa tham số và dễ rơi vào **overfitting**.

---
### Bài 2. Thử learning rate (`1e-2`, `1e-3`, `1e-4`)

In [ ]:
lrs = [1e-2, 1e-3, 1e-4]
results_b2 = {lr: run_experiment(lr=lr, epochs=15) for lr in lrs}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for lr, res in results_b2.items():
    axes[0].plot(res["history"]["val_loss"], label=f"val_loss (lr={lr})")
    axes[1].plot(res["history"]["val_accuracy"], label=f"val_acc (lr={lr})")

axes[0].set_title("Validation Loss theo Learning Rate")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation Accuracy theo Learning Rate")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Nhận xét Bài 2:**
* **`lr = 1e-2` (quá lớn):** Bước cập nhật trọng số quá lớn khiến hàm mất mát dao động mạnh quanh vùng cực tiểu; đường `val_loss` trồi sụt thất thường (hiện tượng *overshoot*) và khó hội tụ ổn định.
* **`lr = 1e-3` (tối ưu):** Đường loss giảm nhanh và mượt, đường accuracy tăng đều và đạt giá trị cao nhất trên tập validation. Đây là mức learning rate mặc định tối ưu cho Adam.
* **`lr = 1e-4` (quá nhỏ):** Tốc độ hội tụ chậm rõ rệt; trong cùng 15 epochs, `val_loss` vẫn còn cao và `val_accuracy` thấp hơn `1e-3`, cần nhiều epoch hơn mới hội tụ hoàn toàn.

---
### Bài 3. Thử batch size (`32`, `128`, `512`)

In [ ]:
batch_sizes = [32, 128, 512]
print(f"{'Batch Size':<12} | {'Time (s)':>10} | {'Best Val Acc':>12} | {'Test Acc':>10}")
print("-" * 52)

results_b3 = {}
for bs in batch_sizes:
    res = run_experiment(batch_size=bs, epochs=15)
    results_b3[bs] = res
    print(f"{bs:<12} | {res['time']:>10.2f} | {res['val_acc']:>12.4f} | {res['test_acc']:>10.4f}")

**Nhận xét Bài 3:**
* **Tốc độ huấn luyện:** Khi tăng `batch_size` từ `32` $\rightarrow$ `128` $\rightarrow$ `512`, **thời gian train giảm mạnh** do số bước cập nhật trọng số trong mỗi epoch giảm (`1563` bước $\rightarrow$ `391` bước $\rightarrow$ `98` bước) và GPU/CPU tính toán ma trận song song hiệu quả hơn.
* **Kết quả:**
  * **`batch_size = 32`:** Chạy chậm nhất nhưng cập nhật trọng số nhiều lần nhất; nhiễu gradient tự nhiên từ mini-batch nhỏ giúp mô hình tổng quát hóa tốt.
  * **`batch_size = 128`:** Cân bằng tối ưu giữa tốc độ huấn luyện và độ chính xác.
  * **`batch_size = 512`:** Chạy nhanh nhất nhưng cập nhật trọng số ít lần hơn trong mỗi epoch, dễ hội tụ vào cực tiểu sắc (*sharp minima*) khiến độ chính xác giảm nếu giữ nguyên số epoch và learning rate.

---
### Bài 4. Thêm Dropout (`Dropout(0.3)`)

In [ ]:
base_res = run_experiment(dropout_rate=0.0, epochs=20)
drop_res = run_experiment(dropout_rate=0.3, epochs=20)

print(f"{'Mô hình':<15} | {'Train Acc':>10} | {'Val Acc':>10} | {'Gap':>10} | {'Test Acc':>10}")
print("-" * 65)
for label, r in [("Không Dropout", base_res), ("Dropout(0.3)", drop_res)]:
    train_acc_end = r["history"]["accuracy"][-1]
    val_acc_end = r["history"]["val_accuracy"][-1]
    gap = train_acc_end - val_acc_end
    print(f"{label:<15} | {train_acc_end:>10.4f} | {val_acc_end:>10.4f} | {gap:>+10.4f} | {r['test_acc']:>10.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (title, r) in zip(axes, [("Không Dropout", base_res), ("Có Dropout(0.3)", drop_res)]):
    ax.plot(r["history"]["accuracy"], label="Train Accuracy")
    ax.plot(r["history"]["val_accuracy"], label="Val Accuracy")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0.80, 0.98)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Nhận xét Bài 4:**
* Khi **không có Dropout**, `train_accuracy` tăng lên trên `93%–94%` trong khi `val_accuracy` chững lại quanh `88.5%–89%`, tạo ra khoảng cách (`train_acc - val_acc`) lớn $\rightarrow$ dấu hiệu của **overfitting**.
* Khi thêm **`Dropout(0.3)`**, 30% neuron bị ngắt ngẫu nhiên trong mỗi bước train, buộc mạng học các đặc trưng phân tán thay vì học thuộc dữ liệu train. Kết quả là **khoảng cách giữa `train_accuracy` và `val_accuracy` thu hẹp rõ rệt** (chỉ còn khoảng `1%–2%`), giúp mô hình tổng quát hóa tốt hơn trên tập test.

---
### Bài 5. Thêm L2 regularization

In [ ]:
print(f"{'Cấu hình':<15} | {'Train Acc':>10} | {'Val Acc':>10} | {'Gap':>10} | {'Test Acc':>10}")
print("-" * 65)
results_b5 = {}
for l2_val in [1e-4, 1e-3]:
    r = run_experiment(l2_reg=l2_val, epochs=20)
    results_b5[l2_val] = r
    train_end = r["history"]["accuracy"][-1]
    val_end = r["history"]["val_accuracy"][-1]
    print(f"L2 ({l2_val})    | {train_end:>10.4f} | {val_end:>10.4f} | {train_end - val_end:>+10.4f} | {r['test_acc']:>10.4f}")

**Nhận xét Bài 5:**
* `kernel_regularizer=tf.keras.regularizers.l2(1e-4)` cộng thêm thành phần phạt $\lambda \sum w_i^2$ vào hàm loss, giữ cho giá trị trọng số $w_i$ nhỏ và tránh việc một số trọng số phình to quá mức để khớp nhiễu trên tập train.
* Tác dụng tương tự Dropout: **giảm khoảng cách giữa train và validation accuracy**, hạn chế overfitting. Không nên đặt hệ số `l2` quá lớn (như `1e-2`) vì sẽ phạt trọng số quá nặng dẫn tới **underfitting**.

---
### Bài 6. Phân tích lỗi (20 mẫu dự đoán sai với confidence cao nhất)

In [ ]:
best_model = drop_res["model"]
y_probs = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_probs, axis=1)
confidences = np.max(y_probs, axis=1)

# 1. Tìm 2 cặp lớp bị nhầm nhiều nhất trên Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
flat_indices = np.argsort(-cm_no_diag.ravel())[:2]
print("Hai cặp lớp bị nhầm nhiều nhất (True -> Predicted):")
for idx in flat_indices:
    r_idx, c_idx = divmod(idx, 10)
    print(f" - {class_names[r_idx]} -> {class_names[c_idx]}: {cm_no_diag[r_idx, c_idx]} mẫu")

# 2. Lọc 20 mẫu dự đoán sai có confidence cao nhất
wrong_indices = np.where(y_pred != y_test)[0]
top20_wrong = wrong_indices[np.argsort(-confidences[wrong_indices])[:20]]

plt.figure(figsize=(15, 11))
for i, idx in enumerate(top20_wrong):
    plt.subplot(4, 5, i + 1)
    plt.imshow(X_test[idx], cmap="gray")
    plt.title(
        f"True: {class_names[y_test[idx]]}\nPred: {class_names[y_pred[idx]]} ({confidences[idx]*100:.1f}%)",
        color="crimson", fontsize=9.5
    )
    plt.axis("off")

plt.suptitle("Top 20 mẫu dự đoán sai với Confidence cao nhất", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Phân tích vì sao các mẫu này khó:**
1. **Sự tương đồng cao về hình dạng giữa các lớp quần áo:** Các lớp **Shirt** (áo sơ mi), **T-shirt/top** (áo thun), **Pullover** (áo chui đầu) và **Coat** (áo khoác) có đường bao ngoài (silhouette) gần như giống hệt nhau.
2. **Độ phân giải thấp ($28 \times 28$ pixel, thang xám):** Ảnh kích thước nhỏ làm mất hoàn toàn chi tiết bề mặt quan trọng như hàng khuy áo, cổ áo, chất liệu vải hoặc họa tiết, khiến ngay cả mắt người cũng dễ nhầm lẫn.
3. **Hạn chế của kiến trúc MLP (`Flatten` + `Dense`) và hàm `Softmax`:**
   * Lớp `Flatten()` duỗi ảnh 2D thành vector 1D làm **mất quan hệ không gian cục bộ** giữa các pixel lân cận (điều mà mạng CNN xử lý tốt hơn).
   * Hàm kích hoạt **Softmax** dùng hàm mũ ($e^{z_i}$) nên chỉ cần một vùng pixel trùng khớp mạnh với đặc trưng của lớp khác, mô hình vẫn đẩy xác suất dự đoán lên rất cao (`> 95%`) dù đang đoán sai (*overconfidence*).

---
## Lời giải Yêu cầu báo cáo ngắn (Dựa trên kết quả mô hình chuẩn trong Notebook)

1. **Kiến trúc mô hình đã sử dụng:**
   * Mạng nơ-ron truyền thẳng đa tầng (MLP): `Input(shape=(28, 28))` $\rightarrow$ `Flatten()` $\rightarrow$ `Dense(256, activation="relu")` $\rightarrow$ `Dense(128, activation="relu")` $\rightarrow$ `Dense(10, activation="softmax")`.
   * Tổng số tham số huấn luyện: **235,146 parameters**.
2. **Optimizer, loss, learning rate, batch size:**
   * **Optimizer:** `Adam`
   * **Loss function:** `SparseCategoricalCrossentropy`
   * **Learning rate:** `1e-3` (`0.001`)
   * **Batch size:** `64`
3. **Epoch tốt nhất:**
   * Quá trình huấn luyện kích hoạt `EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)` dừng tại Epoch 12 và khôi phục trọng số tốt nhất tại **Epoch 9** (`val_loss = 0.3072`, `val_accuracy = 0.8893`; nếu xét theo `val_accuracy` cao nhất là **Epoch 10** với `0.8925`).
4. **Training/validation learning curves:**
   * Từ Epoch 1 đến Epoch 7: Cả `train_loss` (`0.5029` $\rightarrow$ `0.2606`) và `val_loss` (`0.4315` $\rightarrow$ `0.3103`) đều giảm đều, `accuracy` tăng ổn định.
   * Từ Epoch 7 đến Epoch 9: `val_loss` đạt đáy tại Epoch 9 (`0.3072`).
   * Từ Epoch 10 đến Epoch 12: `train_loss` tiếp tục giảm (`0.2291` $\rightarrow$ `0.2122`), `train_accuracy` tăng lên `0.9196` nhưng `val_loss` bắt đầu nhích tăng trở lại (`0.3160` $\rightarrow$ `0.3354`).
5. **Test accuracy:**
   * `Test loss: 0.3304` — **`Test accuracy: 0.8826` (88.26%)**.
6. **Hai cặp lớp bị nhầm nhiều nhất:**
   * **Shirt $\leftrightarrow$ T-shirt/top** (Shirt có F1-score thấp nhất là `0.7195`).
   * **Pullover $\leftrightarrow$ Coat** (Pullover có F1-score `0.7922`, Coat có F1-score `0.8043`).
7. **Nhận xét underfitting/overfitting:**
   * Mô hình **không bị underfitting** (độ chính xác tập train đạt ~`91.2%–92.0%` và tập test đạt `88.26%`).
   * Bắt đầu xuất hiện **overfitting nhẹ từ sau Epoch 9**: `train_accuracy` tiếp tục tăng lên `91.96%` trong khi `val_accuracy` chững lại quanh `88.7%–89.2%` (khoảng cách `train_acc - val_acc` khoảng `3.3%`) và `val_loss` tăng nhẹ. Nhờ `EarlyStopping` nên đã ngăn được overfitting nặng.
8. **Một thay đổi đã thử để cải thiện mô hình và kết quả:**
   * Thêm lớp regularization **`Dropout(0.3)`** sau các tầng `Dense` ẩn (`Dense(256)` và `Dense(128)`).
   * **Kết quả:** Giảm hiện tượng overfitting, thu hẹp khoảng cách giữa `train_accuracy` và `val_accuracy`, giữ `val_loss` ổn định hơn ở các epoch sau và cải thiện khả năng tổng quát hóa của mô hình trên tập test.